<h4>Prepare Dataset</h4>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import json
from pathlib import Path

from datasets import load_dataset
from transformers import AutoTokenizer


# Configuration
DATASET_NAME = "EleutherAI/the_pile_deduplicated"
DATASET_SPLIT = "train"

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

MAX_SAMPLES = None  # Set an integer to limit the number of prompts.
OUTPUT_FILE = "data/attacker/pile_attacker_train.jsonl"

MIN_TOKENS = 40
MAX_TOKENS = 256

MIN_CHARACTERS = 100
MAX_CHARACTERS = 6000


# Helpers
def clean_text(value):
    if value is None:
        return ""

    return (
        str(value)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )


def normalize_for_deduplication(text):
    return " ".join(text.lower().split())


def render_chat(tokenizer, text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        add_generation_prompt=True,
        tokenize=False,
    )


# Main
def main():
    print(f"Loading tokenizer: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    print(f"Streaming dataset: {DATASET_NAME}")

    # This dataset currently uses the default configuration.
    # Do not pass "all" as a configuration name.
    dataset = load_dataset(
        DATASET_NAME,
        split=DATASET_SPLIT,
        streaming=True,
    )

    saved = 0
    scanned = 0
    seen = set()

    Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8",
    ) as output_file:

        for example in dataset:
            scanned += 1

            text = clean_text(example.get("text", ""))

            if not text:
                continue

            if len(text) < MIN_CHARACTERS:
                continue

            if len(text) > MAX_CHARACTERS:
                continue

            normalized_text = normalize_for_deduplication(text)

            if normalized_text in seen:
                continue

            try:
                rendered_prompt = render_chat(
                    tokenizer,
                    text,
                )
            except Exception as error:
                print(
                    f"Skipping row {scanned}: "
                    f"chat-template error: {error}"
                )
                continue

            tokenized = tokenizer(
                rendered_prompt,
                add_special_tokens=False,
                truncation=False,
            )

            token_length = len(tokenized["input_ids"])

            if token_length < MIN_TOKENS:
                continue

            if token_length > MAX_TOKENS:
                continue

            seen.add(normalized_text)

            record = {
                "text": text,
                "prompt_token_length": token_length,
                "source": DATASET_NAME,
            }

            output_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

            saved += 1

            if saved % 100 == 0:
                print(
                    f"Saved {saved} | "
                    f"scanned={scanned} | "
                    f"last_token_length={token_length}"
                )

            if MAX_SAMPLES is not None and saved >= MAX_SAMPLES:
                break

    print("=" * 70)
    print(f"Saved examples : {saved}")
    print(f"Rows scanned   : {scanned}")
    print(f"Output file    : {OUTPUT_FILE}")
    print("=" * 70)

    if MAX_SAMPLES is not None and saved < MAX_SAMPLES:
        print(
            f"Warning: requested {MAX_SAMPLES} examples, "
            f"but only saved {saved}."
        )


if __name__ == "__main__":
    main()


<h4>Capture_Activations<h4></h4

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""Generate clean auxiliary activations at a selected operator input."""

import argparse
import gc
import hashlib
import json
import os
import random
import time
from pathlib import Path
from typing import Any

import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


# CONFIGURATION
INPUT_JSONL = "data/attacker/pile_attacker_train.jsonl"
DEFAULT_OUTPUT_ROOT = Path("results/attacker_clean_data")

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# Select the capture point here, or override it with command-line arguments.
CAPTURE_LAYER = 0

CAPTURE_POINT = "lm_head_input_after_final_norm"

# Supported values:
#   "qkv_input_after_input_layernorm"
#   "o_proj_input"
#   "gate_up_input_after_post_attention_layernorm"
#   "lm_head_input_after_final_norm"

CONDITION = "plain_t3"
MASKED = False

VALID_FRACTION = 0.10
SEED = 42

MIN_TOKENS = 40
MAX_TOKENS = 256

EXPECTED_HIDDEN_SIZE = 3072
MODEL_DTYPE = torch.float32
SAVE_DTYPE = torch.float32

LOG_EVERY = 25


# CAPTURE-POINT DEFINITIONS
SUPPORTED_CAPTURE_POINTS = {
    "qkv_input_after_input_layernorm",
    "o_proj_input",
    "gate_up_input_after_post_attention_layernorm",
    "lm_head_input_after_final_norm",
}


def capture_point_short_name(capture_point: str) -> str:
    names = {
        "qkv_input_after_input_layernorm": "qkv",
        "o_proj_input": "o",
        "gate_up_input_after_post_attention_layernorm": "gate_up",
        "lm_head_input_after_final_norm": "lm_head",
    }
    return names[capture_point]


def default_output_dir(
    capture_layer: int,
    capture_point: str,
) -> Path:
    short_name = capture_point_short_name(capture_point)

    if capture_point == "lm_head_input_after_final_norm":
        return DEFAULT_OUTPUT_ROOT / "Final_LMHead"

    return DEFAULT_OUTPUT_ROOT / f"Layer{capture_layer}_{short_name}"


def resolve_capture_target(
    model: Any,
    capture_layer: int,
    capture_point: str,
):
    """
    Return:
        target_module,
        op_names,
        effective_layer,
        human-readable description
    """
    if capture_point not in SUPPORTED_CAPTURE_POINTS:
        raise ValueError(
            f"Unsupported capture point: {capture_point!r}. "
            f"Supported values: {sorted(SUPPORTED_CAPTURE_POINTS)}"
        )

    num_layers = len(model.model.layers)

    if capture_point != "lm_head_input_after_final_norm":
        if not 0 <= capture_layer < num_layers:
            raise ValueError(
                f"CAPTURE_LAYER must be in [0, {num_layers - 1}], "
                f"got {capture_layer}"
            )

        layer = model.model.layers[capture_layer]

    if capture_point == "qkv_input_after_input_layernorm":
        return (
            layer.self_attn.q_proj,
            ["q_proj", "k_proj", "v_proj"],
            capture_layer,
            (
                f"layer {capture_layer} shared QKV input "
                "after input_layernorm"
            ),
        )

    if capture_point == "o_proj_input":
        return (
            layer.self_attn.o_proj,
            ["o_proj"],
            capture_layer,
            f"layer {capture_layer} o_proj input",
        )

    if capture_point == "gate_up_input_after_post_attention_layernorm":
        return (
            layer.mlp.gate_proj,
            ["gate_proj", "up_proj"],
            capture_layer,
            (
                f"layer {capture_layer} shared Gate/Up input "
                "after post_attention_layernorm"
            ),
        )

    # lm_head_input_after_final_norm
    return (
        model.lm_head,
        ["lm_head"],
        None,
        "final lm_head input after final model norm",
    )


# HELPERS
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clean_text(value) -> str:
    if value is None:
        return ""

    return (
        str(value)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )


def stable_sample_id(index: int, text: str) -> str:
    digest = hashlib.sha256(
        f"{index}\0{text}".encode("utf-8")
    ).hexdigest()

    return f"pile_{index:06d}_{digest[:16]}"


def render_chat(tokenizer, text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def append_jsonl(path: Path, obj: dict) -> None:
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(obj, ensure_ascii=False) + "\n")
        file.flush()


def load_jsonl(path: str):
    rows = []

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON at line {line_number}: {error}"
                ) from error

            text = clean_text(obj.get("text", ""))

            if not text:
                continue

            rows.append(
                {
                    "source_line": line_number,
                    "text": text,
                    "declared_prompt_token_length": obj.get(
                        "prompt_token_length"
                    ),
                    "source": obj.get("source"),
                }
            )

    return rows


def validate_existing_sample(
    sample_path: Path,
    sample_id: str,
    expected_capture_point: str,
    expected_layer,
) -> dict:
    obj = torch.load(sample_path, map_location="cpu")

    if obj.get("sample_id") != sample_id:
        raise RuntimeError(
            f"Existing file has wrong sample_id: {sample_path}"
        )

    hidden = obj.get("hidden")

    if not torch.is_tensor(hidden):
        raise RuntimeError(
            f"Existing file has no valid hidden tensor: {sample_path}"
        )

    if hidden.ndim != 2:
        raise RuntimeError(
            f"Expected [T,D], got {tuple(hidden.shape)} in {sample_path}"
        )

    if hidden.shape[-1] != EXPECTED_HIDDEN_SIZE:
        raise RuntimeError(
            f"Expected hidden size {EXPECTED_HIDDEN_SIZE}, "
            f"got {hidden.shape[-1]} in {sample_path}"
        )

    if torch.isnan(hidden).any().item():
        raise RuntimeError(f"NaN detected in {sample_path}")

    if torch.isinf(hidden).any().item():
        raise RuntimeError(f"Inf detected in {sample_path}")

    meta = obj.get("meta", {})

    if meta.get("capture_point") != expected_capture_point:
        raise RuntimeError(
            "Existing sample belongs to a different capture point: "
            f"{meta.get('capture_point')!r} != "
            f"{expected_capture_point!r}"
        )

    if meta.get("layer_idx") != expected_layer:
        raise RuntimeError(
            "Existing sample belongs to a different layer: "
            f"{meta.get('layer_idx')!r} != {expected_layer!r}"
        )

    return obj


# GENERIC OPERATOR-INPUT CAPTURE
class OperatorInputCapture:
    """
    Captures the input tensor received by the selected module.

    A forward pre-hook is used because plain T3 exposes the operator input.
    """

    def __init__(
        self,
        target_module,
        description: str,
    ):
        self.tensor = None
        self.call_count = 0
        self.description = description

        self.handle = target_module.register_forward_pre_hook(
            self._pre_hook
        )

    def _pre_hook(self, module, inputs):
        del module

        self.call_count += 1

        if not inputs:
            raise RuntimeError(
                f"{self.description} pre-hook received no inputs"
            )

        hidden = inputs[0]

        if isinstance(hidden, (tuple, list)):
            hidden = hidden[0]

        if not torch.is_tensor(hidden):
            raise RuntimeError(
                f"{self.description} input is not a tensor: "
                f"{type(hidden)}"
            )

        self.tensor = (
            hidden
            .detach()
            .to(device="cpu", dtype=SAVE_DTYPE)
            .contiguous()
            .clone()
        )

    def reset(self):
        self.tensor = None
        self.call_count = 0

    def close(self):
        self.handle.remove()


# MAIN
def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        type=str,
        default=INPUT_JSONL,
    )
    parser.add_argument(
        "--output-dir",
        type=str,
        default=None,
        help=(
            "Output directory. When omitted, a directory is generated "
            "from the selected layer and capture point."
        ),
    )
    parser.add_argument(
        "--capture-layer",
        type=int,
        default=CAPTURE_LAYER,
        help=(
            "Transformer layer index. Ignored for the final lm_head input."
        ),
    )
    parser.add_argument(
        "--capture-point",
        type=str,
        default=CAPTURE_POINT,
        choices=sorted(SUPPORTED_CAPTURE_POINTS),
    )
    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Optional sample limit for a smoke test.",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
    )

    args, unknown = parser.parse_known_args()

    if unknown:
        print(f"Ignoring Jupyter arguments: {unknown}")

    capture_layer = int(args.capture_layer)
    capture_point = str(args.capture_point)

    output_dir = (
        Path(args.output_dir)
        if args.output_dir
        else default_output_dir(capture_layer, capture_point)
    )

    samples_dir = output_dir / "samples"
    train_manifest = output_dir / "train_manifest.jsonl"
    valid_manifest = output_dir / "valid_manifest.jsonl"
    failed_rows = output_dir / "failed_rows.jsonl"
    dataset_meta = output_dir / "dataset_meta.json"

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required for the recommended run."
        )

    set_seed(SEED)

    output_dir.mkdir(parents=True, exist_ok=True)
    samples_dir.mkdir(parents=True, exist_ok=True)

    rows = load_jsonl(args.input)

    if args.limit is not None:
        rows = rows[: max(0, int(args.limit))]

    if not rows:
        raise RuntimeError("No valid input rows were found")

    print("=" * 88)
    print("LOCAL AUXILIARY ACTIVATION GENERATOR")
    print("=" * 88)
    print(f"Input JSONL          : {args.input}")
    print(f"Rows loaded          : {len(rows)}")
    print(f"Output directory     : {output_dir}")
    print(f"Model                : {MODEL_NAME}")
    print(f"Model dtype          : {MODEL_DTYPE}")
    print(f"Save dtype           : {SAVE_DTYPE}")
    print(f"Capture layer        : {capture_layer}")
    print(f"Capture point        : {capture_point}")
    print(f"Condition            : {CONDITION}")
    print(f"Masked               : {MASKED}")
    print(f"Validation fraction  : {VALID_FRACTION}")
    print("=" * 88)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    print("Loading victim model on GPU...", flush=True)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
    ).to("cuda").eval()

    (
        target_module,
        op_names,
        effective_layer,
        capture_description,
    ) = resolve_capture_target(
        model=model,
        capture_layer=capture_layer,
        capture_point=capture_point,
    )

    print(f"Resolved target       : {capture_description}")
    print(f"Operator names        : {op_names}")
    print(f"Effective layer       : {effective_layer}")
    print("=" * 88)

    capture = OperatorInputCapture(
        target_module=target_module,
        description=capture_description,
    )

    successful_records = []
    failed_count = 0

    if failed_rows.exists():
        failed_rows.unlink()

    try:
        for index, row in enumerate(
            tqdm(rows, desc="Generating activations")
        ):
            text = row["text"]
            sample_id = stable_sample_id(index, text)
            sample_path = samples_dir / f"{sample_id}.pt"

            try:
                if sample_path.exists() and not args.overwrite:
                    existing = validate_existing_sample(
                        sample_path=sample_path,
                        sample_id=sample_id,
                        expected_capture_point=capture_point,
                        expected_layer=effective_layer,
                    )

                    hidden = existing["hidden"]
                    token_length = int(hidden.shape[0])

                    successful_records.append(
                        {
                            "sample_id": sample_id,
                            "sample_path": str(sample_path),
                            "text": text,
                            "prompt_token_length": token_length,
                            "hidden_size": int(hidden.shape[1]),
                            "source_line": row["source_line"],
                            "source": row["source"],
                        }
                    )
                    continue

                rendered = render_chat(tokenizer, text)

                encoded = tokenizer(
                    rendered,
                    return_tensors="pt",
                    add_special_tokens=False,
                    truncation=False,
                )

                token_length = int(encoded["input_ids"].shape[1])

                if not MIN_TOKENS <= token_length <= MAX_TOKENS:
                    raise ValueError(
                        f"Token length {token_length} is outside "
                        f"[{MIN_TOKENS}, {MAX_TOKENS}]"
                    )

                declared = row["declared_prompt_token_length"]

                if (
                    declared is not None
                    and int(declared) != token_length
                ):
                    raise ValueError(
                        "Token-length mismatch: "
                        f"declared={declared}, actual={token_length}"
                    )

                encoded = {
                    key: value.to("cuda")
                    for key, value in encoded.items()
                }

                capture.reset()

                with torch.inference_mode():
                    _ = model(
                        **encoded,
                        use_cache=False,
                        output_attentions=False,
                        output_hidden_states=False,
                        return_dict=True,
                    )

                if capture.call_count != 1:
                    raise RuntimeError(
                        f"Expected selected hook once, "
                        f"got {capture.call_count}"
                    )

                hidden = capture.tensor

                if hidden is None:
                    raise RuntimeError(
                        "No selected operator input was captured"
                    )

                if hidden.ndim != 3:
                    raise RuntimeError(
                        f"Expected [B,T,D], got {tuple(hidden.shape)}"
                    )

                if hidden.shape[0] != 1:
                    raise RuntimeError(
                        f"Expected batch size 1, got {hidden.shape[0]}"
                    )

                hidden = hidden[0].contiguous()

                if int(hidden.shape[0]) != token_length:
                    raise RuntimeError(
                        "Captured sequence length mismatch: "
                        f"captured={hidden.shape[0]}, "
                        f"tokens={token_length}"
                    )

                if int(hidden.shape[1]) != EXPECTED_HIDDEN_SIZE:
                    raise RuntimeError(
                        f"Expected hidden size {EXPECTED_HIDDEN_SIZE}, "
                        f"got {hidden.shape[1]}"
                    )

                if torch.isnan(hidden).any().item():
                    raise RuntimeError(
                        "Captured tensor contains NaN"
                    )

                if torch.isinf(hidden).any().item():
                    raise RuntimeError(
                        "Captured tensor contains Inf"
                    )

                payload = {
                    "sample_id": sample_id,
                    "hidden": hidden,
                    "text": text,
                    "meta": {
                        "model_name": MODEL_NAME,
                        "condition": CONDITION,
                        "masked": MASKED,
                        "capture_tensor": "operator_input",
                        "capture_point": capture_point,
                        "capture_description": capture_description,
                        "layer_idx": effective_layer,
                        "op_names": op_names,
                        "phase": "prefill",
                        "prompt_token_length": token_length,
                        "hidden_size": int(hidden.shape[1]),
                        "stored_shape": tuple(
                            int(x) for x in hidden.shape
                        ),
                        "dtype": str(hidden.dtype),
                        "source_line": row["source_line"],
                        "source": row["source"],
                        "generated_locally": True,
                        "created_unix_time": time.time(),
                    },
                }

                torch.save(payload, sample_path)

                successful_records.append(
                    {
                        "sample_id": sample_id,
                        "sample_path": str(sample_path),
                        "text": text,
                        "prompt_token_length": token_length,
                        "hidden_size": int(hidden.shape[1]),
                        "source_line": row["source_line"],
                        "source": row["source"],
                    }
                )

                if (
                    len(successful_records) == 1
                    or len(successful_records) % LOG_EVERY == 0
                ):
                    print(
                        f"\nSaved {len(successful_records)} | "
                        f"sample={sample_id} | "
                        f"shape={tuple(hidden.shape)} | "
                        f"dtype={hidden.dtype}"
                    )

                del encoded
                del hidden

            except Exception as error:
                failed_count += 1

                append_jsonl(
                    failed_rows,
                    {
                        "index": index,
                        "source_line": row["source_line"],
                        "sample_id": sample_id,
                        "capture_point": capture_point,
                        "layer_idx": effective_layer,
                        "error": repr(error),
                    },
                )

                print(
                    f"\n[FAILED] index={index} "
                    f"sample={sample_id}: {error}",
                    flush=True,
                )

            if (index + 1) % 50 == 0:
                gc.collect()
                torch.cuda.empty_cache()

    finally:
        capture.close()

    if len(successful_records) < 2:
        raise RuntimeError(
            "Fewer than two samples were generated successfully"
        )

    shuffled = list(successful_records)
    rng = random.Random(SEED)
    rng.shuffle(shuffled)

    valid_count = max(
        1,
        int(round(len(shuffled) * VALID_FRACTION)),
    )

    valid_records = shuffled[:valid_count]
    train_records = shuffled[valid_count:]

    if not train_records:
        raise RuntimeError("Training split is empty")

    with open(train_manifest, "w", encoding="utf-8") as file:
        for record in train_records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    with open(valid_manifest, "w", encoding="utf-8") as file:
        for record in valid_records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    meta = {
        "model_name": MODEL_NAME,
        "condition": CONDITION,
        "masked": MASKED,
        "capture_layer_requested": capture_layer,
        "capture_layer": effective_layer,
        "capture_point": capture_point,
        "capture_description": capture_description,
        "capture_tensor": "operator_input",
        "op_names": op_names,
        "phase": "prefill",
        "hidden_size": EXPECTED_HIDDEN_SIZE,
        "model_dtype": str(MODEL_DTYPE),
        "save_dtype": str(SAVE_DTYPE),
        "input_jsonl": os.path.abspath(args.input),
        "output_directory": str(output_dir),
        "samples_directory": str(samples_dir),
        "train_manifest": str(train_manifest),
        "valid_manifest": str(valid_manifest),
        "seed": SEED,
        "validation_fraction": VALID_FRACTION,
        "num_input_rows": len(rows),
        "num_successful": len(successful_records),
        "num_failed": failed_count,
        "num_train": len(train_records),
        "num_valid": len(valid_records),
        "min_tokens": MIN_TOKENS,
        "max_tokens": MAX_TOKENS,
        "created_unix_time": time.time(),
    }

    with open(dataset_meta, "w", encoding="utf-8") as file:
        json.dump(meta, file, indent=2)

    print("=" * 88)
    print("ACTIVATION GENERATION COMPLETE")
    print("=" * 88)
    print(f"Capture point      : {capture_point}")
    print(f"Effective layer    : {effective_layer}")
    print(f"Successful samples : {len(successful_records)}")
    print(f"Failed samples     : {failed_count}")
    print(f"Training samples   : {len(train_records)}")
    print(f"Validation samples : {len(valid_records)}")
    print(f"Samples directory  : {samples_dir}")
    print(f"Train manifest     : {train_manifest}")
    print(f"Valid manifest     : {valid_manifest}")
    print(f"Dataset metadata   : {dataset_meta}")
    print(f"Failure log        : {failed_rows}")
    print("=" * 88)


if __name__ == "__main__":
    main()
